# Laboratorium 2: Współbieżność i Równoległość w Pythonie
### Skoroszyt Edukacyjny - Wersja dla Studentów

---

## 1. Wstęp: Koncepcja "Wielu Zadań"

Zanim zaczniemy pisać kod, musimy rozróżnić dwa kluczowe pojęcia:

1. **Współbieżność (Concurrency)**: Wykonywanie wielu zadań "na zmianę". Wyobraź sobie kelnera, który obsługuje 5 stolików. Nie robi wszystkiego naraz, ale szybko przełącza się między nimi. Dla klientów wygląda to, jakby obsługiwał ich równocześnie.
2. **Równoległość (Parallelism)**: Wykonywanie wielu zadań faktycznie w tym samym momencie. To sytuacja, w której mamy 5 kelnerów i każdy obsługuje jeden stolik.

W Pythonie współbieżność realizujemy najczęściej za pomocą **Wątków (Threads)**, a równoległość za pomocą **Procesów (Processes)**.

---

## 2. Wielowątkowość (Threading) - Zadania I/O-bound

Wątki są idealne, gdy program większość czasu spędza na **czekaniu** na odpowiedź z sieci (zapytania HTTP). W tym czasie procesor się nudzi – wątki pozwalają mu wysłać kolejne zapytania, nie czekając na poprzednie.

---

### Demo: Scraping Kalendarza Kulturalnego (Krakow.pl)

**Kod zawarty w poniższych komórkach (analogicznie do plików `lab_2_1_demo.py` oraz `lab_2_2_demo.py`) pozwala na pobieranie tytułów wydarzeń kulturalnych z oficjalnego kalendarium miasta Krakowa (krakow.pl).** 

Przykładowy adres źródłowy: `https://www.krakow.pl/kalendarium/1919,shw,2026-03-20,0,day.html`. 

Demo pokazuje proces pobierania danych z 5 kolejnych stron tego zestawienia:
1. **Wersja sekwencyjna**: Zadanie wykonywane jest krok po kroku, co pozwala zaobserwować sumaryczny czas oczekiwania na każde z zapytań HTTP z osobna (wysoki koszt operacji wejścia/wyjścia).
2. **Optymalizacja**: Kod zostaje zmodyfikowany z użyciem modułu `concurrent.futures`, wykorzystując `ThreadPoolExecutor`.

Dzięki temu zapytania sieciowe są wysyłane równolegle, co drastycznie skraca czas całkowity działania programu, demonstrując praktyczną przewagę wielowątkowości w zadaniach typu **I/O-bound** (zależnych od odpowiedzi sieciowej).

In [ ]:
import requests
from bs4 import BeautifulSoup
import time

def download_site(url):
    """Pobiera jedną stronę i wyciąga tytuły wydarzeń."""
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')
    event_titles = [item.text.strip() for item in soup.select('.item__link h3')]
    return event_titles

def run_sequential_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]
    
    print(f"Rozpoczynam pobieranie SEKWENCYJNE 5 stron...")
    start = time.time()
    
    all_titles = []
    for url in sites:
        all_titles.extend(download_site(url))
        
    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")
        
    print(f"\nCzas wykonania: {time.time() - start:.2f}s")

run_sequential_demo()

In [ ]:
import concurrent.futures

def run_threaded_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]
    
    print(f"Rozpoczynam pobieranie WIELOWĄTKOWE 5 stron...")
    start = time.time()
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
        results = list(executor.map(download_site, sites))
    
    all_titles = [title for sublist in results for title in sublist]
    
    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")
        
    print(f"\nCzas wykonania (wątki): {time.time() - start:.2f}s")

run_threaded_demo()

--- 
## 3. Synchronizacja: Problem Hazardu i Lock

Gdy wiele wątków próbuje zmieniać tę samą zmienną w tym samym momencie (np. saldo na koncie), dochodzi do tzw. **Race Condition** (wyścigu). Rozwiązaniem jest **Lock** (blokada).

In [ ]:
import threading

class BankAccount:
    def __init__(self):
        self.balance = 0
        self.lock = threading.Lock()

    def deposit(self, amount):
        with self.lock:
            current = self.balance
            time.sleep(0.0001) # Symulacja opóźnienia
            self.balance = current + amount

account = BankAccount()
with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
    executor.map(lambda _: account.deposit(1), range(100))
    
print(f"Saldo końcowe: {account.balance} zł (oczekiwano: 100)")

--- 
## 4. Wieloprocesowość (Multiprocessing) - Zadania CPU-bound

Kiedy musimy wykonać ciężkie obliczenia matematyczne (np. szukanie liczb pierwszych), wątki nam nie pomogą. Musimy użyć osobnych procesów.

**Ważne (macOS/Windows)**: Ze względu na metodę `spawn` startu procesów, funkcje pomocnicze (jak `find_primes`) muszą znajdować się w zewnętrznym pliku `.py` (tutaj: `lab2_functions.py`) i być importowane.

In [ ]:
import multiprocessing
import time
# Importujemy funkcję z oddzielnego pliku, aby uniknąć błędu spawn na macOS
from lab2_functions import find_primes

def run_primes_demo():
    cores = multiprocessing.cpu_count()
    print(f"Praca na {cores} procesach (rdzeniach)...")
    start = time.time()
    
    limit = 1_000_000
    chunk = limit // cores
    ranges = [(i, i + chunk) for i in range(0, limit, chunk)]

    with multiprocessing.Pool(processes=cores) as pool:
        results = pool.starmap(find_primes, ranges)
    
    print(f"Zakończono w czasie {time.time() - start:.2f}s.")

if __name__ == "__main__":
    run_primes_demo()

---
# Zadania do samodzielnego wykonania

Poniższe zadania należy zrealizować w oparciu o wiedzę zdobytą na laboratoriach oraz instrukcje zawarte w pliku PDF.

Kacper Kaszuba Lab4

### Zadanie 1 (Threading)
Przy użyciu publicznego API **Cat Facts** (`https://catfact.ninja/fact`), które zwraca przy każdym wywołaniu losowy fakt na temat kotów:
1. Pobierz sekwencyjnie 20 faktów i zmierz czas całkowitego działania programu.
2. Zmodyfikuj kod, aby wysyłać zapytania wielowątkowo przy użyciu `ThreadPoolExecutor`.
3. Porównaj czasy wykonania.

*Podpowiedź: Użyj `requests.get(URL).json().get('fact')`*

In [1]:
# Miejsce na rozwiązanie zadania 1
import requests
import time
import concurrent.futures
from charset_normalizer.api import logger
from concurrent.futures import ThreadPoolExecutor
import json
import logging
import requests
# Zadanie z kotami


CAT_API_URL = 'https://catfact.ninja/fact'
logging.basicConfig(level=logging.INFO, format='%(message)s')


def fetch_single_fact():
    try:
        ciekawostka = requests.get(CAT_API_URL).json().get('fact')
        return ciekawostka
    except Exception as e:
        logger.error(f'Bland: {e}')

def fetch_sequential():
    lista_cieakwostek = []
    for i in range(20):
        ciek = fetch_single_fact()

        lista_cieakwostek.append(ciek)
    return lista_cieakwostek

def fetch_concurrently():
    with ThreadPoolExecutor(max_workers=20) as executor:
        wynik = [executor.submit(fetch_single_fact) for _ in range(20)]
        lista_cieakwostek = [w.result() for w in wynik ]

    return lista_cieakwostek

if __name__ == "__main__":
    logging.info(f"--- ROZPOCZĘCIE POBIERANIA SEKWENCYJNEGO ---")
    start_time = time.time()

    lista_ciek_pr = fetch_sequential()

    end_time = time.time()
    duration = end_time - start_time
    
    logging.info(f"--- ZAKOŃCZONO ---")
    for i in enumerate(lista_ciek_pr):
        print(f"{i}")

    logging.info(f"\nCzas wykonania sekwencyjnego: {duration:.2f} sekund")

    
    
    
    logging.info(f"--- ROZPOCZĘCIE POBIERANIA WSPÓŁBIEŻNEGO ---")
    start_time = time.time()

    lista_ciek_pr = fetch_concurrently()

    end_time = time.time()
    duration = end_time - start_time
    
    logging.info(f"--- ZAKOŃCZONO ---")
    for i in enumerate(lista_ciek_pr):
        print(f"{i}")

    logging.info(f"\nCzas wykonania współbieżnego: {duration:.2f} sekund")



--- ROZPOCZĘCIE POBIERANIA SEKWENCYJNEGO ---
--- ZAKOŃCZONO ---

Czas wykonania sekwencyjnego: 3.47 sekund
--- ROZPOCZĘCIE POBIERANIA WSPÓŁBIEŻNEGO ---


(0, 'The Ancient Egyptian word for cat was mau, which means "to see".')
(1, 'A cat sees about 6 times better than a human at night, and needs 1/6 the amount of of light that a human does - it has a layer of extra reflecting cells which absorb light.')
(2, 'When a family cat died in ancient Egypt, family members would mourn by shaving off their eyebrows. They also held elaborate funerals during which they drank wine and beat their breasts. The cat was embalmed with a sculpted wooden mask and the tiny mummy was placed in the family tomb or in a pet cemetery with tiny mummies of mice.')
(3, 'Miacis, the primitive ancestor of cats, was a small, tree-living creature of the late Eocene period, some 45 to 50 million years ago.')
(4, 'People who are allergic to cats are actually allergic to cat saliva or to cat dander. If the resident cat is bathed regularly the allergic people tolerate it better.')
(5, 'Cats spend nearly 1/3 of their waking hours cleaning themselves.')
(6, 'The group of words

--- ZAKOŃCZONO ---

Czas wykonania współbieżnego: 0.38 sekund


(0, 'All cats have three sets of long hairs that are sensitive to pressure - whiskers, eyebrows,and the hairs between their paw pads.')
(1, 'Most cats had short hair until about 100 years ago, when it became fashionable to own cats and experiment with breeding.')
(2, 'Neutering a male cat will, in almost all cases, stop him from spraying (territorial marking), fighting with other males (at least over females), as well as lengthen his life and improve its quality.')
(3, 'A cat usually has about 12 whiskers on each side of its face.')
(4, 'A cat has more bones than a human; humans have 206, but the cat has 230 (some cites list 245 bones, and state that bones may fuse together as the cat ages).')
(5, 'In ancient Egypt, when a family cat died, all family members would shave their eyebrows as a sign of mourning.')
(6, 'Cats have 3 eyelids.')
(7, 'A cat has more bones than a human being; humans have 206 and the cat has 230 bones.')
(8, 'Unlike other cats, lions have a tuft of hair at the end

### Zadanie 2 (Wątki i Kolejka - Producent-Konsument)
Napisz program o strukturze **producent-consumers**:
1. **Producent**: Generuje kolejne liczby naturalne i dodaje je do kolejki (`queue.Queue`).
2. **Konsument 1**: Pobiera z kolejki tylko liczby **parzyste**.
3. **Konsument 2**: Pobiera z kolejki tylko liczby **nieparzyste**.

Użyj wątków do realizacji producenta i obu konsumentów. Program powinien zakończyć się po przetworzeniu określonej puli liczb.

In [ ]:
# Miejsce na rozwiązanie zadania 2
import queue
import threading
import time
import logging
from charset_normalizer.api import logger
import logging
import concurrent.futures
import queue


def producent(q, nb):
    for item in range(1, nb+1):
        q.put(item)
        logger.info(f"Dodano do kolejki item: {item}\n")
        time.sleep(1)
    q.put(None)
    q.put(None)

def konsument(q, nazwa):
    while True:
        item = q.get()
        if item is not None:
            if (item % 2 ==0) and (nazwa=='parzysty') :
                logging.info(f"Konsument {nazwa} przetworzył PARZYSTĄ: {item}\n")
                # continue
            elif (item % 2 ==0) and (nazwa=='nieparzysty') :
                logging.info(f"Liczba: {item} wraca do kolejki\n")
                q.put(item)
                time.sleep(1)
            elif (item % 2 != 0) and (nazwa=='nieparzysty'):
                logging.info(f"Konsument {nazwa} przetworzył NIEPARZYSTĄ: {item}\n")
                # continue
            else:
                logging.info(f"Liczba: {item} wraca do kolejki\n")
                q.put(item)
                time.sleep(1)

        else:
            q.task_done()
            break
    
        

        q.task_done()




if __name__ == "__main__":
    logging.basicConfig(level=logging.INFO)
    q = queue.Queue()

    start_time = time.time()

    threading.Thread(target=producent, args=(q, 10,)).start()
    threading.Thread(target=konsument, args=(q, 'parzysty',)).start()
    threading.Thread(target=konsument, args=(q,'nieparzysty',)).start()

    end_time = time.time()
    duration = end_time - start_time
    logging.info(f"\nCzas wykonania zadania: {duration:.2f} sekund")


Dodano do kolejki item: 1

Liczba: 1 wraca do kolejki


Czas wykonania zadania: 0.00 sekund
Konsument nieparzysty przetworzył NIEPARZYSTĄ: 1



Dodano do kolejki item: 2

Liczba: 2 wraca do kolejki

Konsument parzysty przetworzył PARZYSTĄ: 2

Dodano do kolejki item: 3

Liczba: 3 wraca do kolejki

Konsument nieparzysty przetworzył NIEPARZYSTĄ: 3

Dodano do kolejki item: 4

Liczba: 4 wraca do kolejki

Konsument parzysty przetworzył PARZYSTĄ: 4

Dodano do kolejki item: 5

Liczba: 5 wraca do kolejki

Konsument nieparzysty przetworzył NIEPARZYSTĄ: 5

Dodano do kolejki item: 6

Liczba: 6 wraca do kolejki

Konsument parzysty przetworzył PARZYSTĄ: 6

Dodano do kolejki item: 7

Liczba: 7 wraca do kolejki

Konsument nieparzysty przetworzył NIEPARZYSTĄ: 7

Dodano do kolejki item: 8

Liczba: 8 wraca do kolejki

Konsument parzysty przetworzył PARZYSTĄ: 8

Dodano do kolejki item: 9

Liczba: 9 wraca do kolejki

Konsument nieparzysty przetworzył NIEPARZYSTĄ: 9

Dodano do kolejki item: 10

Liczba: 10 wraca do kolejki

Konsument parzysty przetworzył PARZYSTĄ: 10



### Zadanie 3 (Multiprocessing)
Napisz program, który zrównolegli obliczanie sumy kolejnych stu potęg dla każdej liczby z ciągu liczb naturalnych w dużym zakresie (np. 1 - 10 000).
Użyj modułu `multiprocessing` oraz gotowej funkcji `calculate_power_sum(n)` z pliku `lab2_functions.py`.

Pamiętaj o bezpiecznym uruchamianiu procesów na macOS (`if __name__ == "__main__":`).

In [ ]:
from multiprocessing import Process
import multiprocessing
import time
from lab2_functions import calculate_power_sum

if __name__ == "__main__":
    start_range = 1
    end_range = 10000
    num_processes = 4

    chunk_size = (end_range - start_range) // num_processes


    with multiprocessing.Pool(num_processes) as pool:
        start_time = time.time()

        results = pool.map(calculate_power_sum, range(start_range, end_range + 1), chunk_size)
        # Drukuje Len bo python nie wydrukuje mi takiej ilosci danych 
        # ale dzięki temu widzimy ze zawiera 10000 
        print(len(results))

        end_time = time.time()
        duration = end_time - start_time

        print(f"\nCzas wykonania zadania: {duration:.2f} sekund")

        pool.close()
        pool.join()


10000

Czas wykonania zadania: 0.22 sekund
